### Testing GraphBuilder

#### Node classification - One Graph - Cora

In [3]:
from torch_geometric import datasets
from rewiring.model.GCN import GCN
from rewiring.pipeline.SingleGraphNodeClassificationPipeline import SingleGraphNodeClassificationPipeline
import torch

dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = SingleGraphNodeClassificationPipeline(
    device=device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.set_masks(data=dset)

pipeline.train_validate(
    data=dset[0],
    epochs=50,
    log=True
)

Epoch: [0/50]         Training loss: 1.8123         Validation loss: 1.7797         Accuracy: 0.2823
Epoch: [1/50]         Training loss: 1.7740         Validation loss: 1.7394         Accuracy: 0.3483
Epoch: [2/50]         Training loss: 1.7289         Validation loss: 1.6961         Accuracy: 0.4505
Epoch: [3/50]         Training loss: 1.6793         Validation loss: 1.6514         Accuracy: 0.4234
Epoch: [4/50]         Training loss: 1.6281         Validation loss: 1.6046         Accuracy: 0.4670
Epoch: [5/50]         Training loss: 1.5747         Validation loss: 1.5547         Accuracy: 0.5105
Epoch: [6/50]         Training loss: 1.5184         Validation loss: 1.5026         Accuracy: 0.5766
Epoch: [7/50]         Training loss: 1.4600         Validation loss: 1.4499         Accuracy: 0.6051
Epoch: [8/50]         Training loss: 1.4007         Validation loss: 1.3973         Accuracy: 0.6456
Epoch: [9/50]         Training loss: 1.3405         Validation loss: 1.3445         Accurac

([1.812345266342163,
  1.7739826440811157,
  1.7289272546768188,
  1.679296612739563,
  1.6281131505966187,
  1.574678897857666,
  1.5184344053268433,
  1.4600050449371338,
  1.4006508588790894,
  1.3405157327651978,
  1.2795228958129883,
  1.2187315225601196,
  1.1587097644805908,
  1.1001174449920654,
  1.043433427810669,
  0.9890037775039673,
  0.9368777871131897,
  0.8870725035667419,
  0.8396991491317749,
  0.7948622107505798,
  0.7526412010192871,
  0.71298748254776,
  0.6757296323776245,
  0.6406378149986267,
  0.6076202392578125,
  0.5766947269439697,
  0.5480604767799377,
  0.5218852162361145,
  0.4981125593185425,
  0.4765472114086151,
  0.45697128772735596,
  0.4391060471534729,
  0.4225096106529236,
  0.40685686469078064,
  0.3920102119445801,
  0.3778531551361084,
  0.3645513653755188,
  0.3523039221763611,
  0.3408617377281189,
  0.3299868702888489,
  0.31968939304351807,
  0.3100825548171997,
  0.3012307584285736,
  0.29305970668792725,
  0.2854008078575134,
  0.27813297

#### Node Classification - Multiple Graphs - LRGB PascalVOC-SP

In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.SimpleGCN import SimpleGCN
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.pipeline.NodeClassificationPipeline import NodeClassifiactionPipeline

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
transform = None

# Load the dataset
print("Loading and transforming training dataset...")
train_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print("Load and transformation done!")

val_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='test')
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = train_dataset.num_features
num_classes = train_dataset.num_classes
hidden_size = 64
model = SimpleGCN(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
# model.set_classification_task('graph')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

pipeline = NodeClassifiactionPipeline(
    device = device,
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

num_epochs = 10
print("Started training...")
pipeline.train_validate(
    train_data=train_loader, 
    val_data=val_loader, 
    epochs=num_epochs, 
    log=True
)
print("Training finished!")

print("Starting testing...")
model.eval()
correct = 0
epoch_test_loss = []
sum = len(test_loader.dataset.y)
print(f'Test element count: {sum}')
for batch in test_loader:
    batch = batch.to(device)
    with torch.no_grad():
        out = model(batch.x, batch.edge_index)
        test_loss = criterion(out, batch.y)
        epoch_test_loss.append(test_loss.item())
        _, pred = torch.max(out.data, 1)
        correct += (pred == batch.y).float().sum().item()

acc = correct / sum
test_loss = np.array(epoch_test_loss).mean().item()

print(f"""Loss: {test_loss:.4f} \
        Accuracy: {acc:.4f}""")

Loading and transforming training dataset...
Load and transformation done!
Started training...
Epoch: [0/10]         Training loss: 16.9142         Validation loss: 2.6159         Accuracy: 0.6946
Epoch: [1/10]         Training loss: 1.8308         Validation loss: 1.6221         Accuracy: 0.6963
Epoch: [2/10]         Training loss: 1.5676         Validation loss: 1.5538         Accuracy: 0.6948
Epoch: [3/10]         Training loss: 1.5245         Validation loss: 1.4997         Accuracy: 0.6934
Epoch: [4/10]         Training loss: 1.4973         Validation loss: 1.4960         Accuracy: 0.6964
Epoch: [5/10]         Training loss: 1.4909         Validation loss: 1.4842         Accuracy: 0.6946
Epoch: [6/10]         Training loss: 1.4741         Validation loss: 1.4752         Accuracy: 0.6948
Epoch: [7/10]         Training loss: 1.4704         Validation loss: 1.4646         Accuracy: 0.6963
Epoch: [8/10]         Training loss: 1.4543         Validation loss: 1.4589         Accuracy: 0.

#### Graph Classification - Proteins, Mutag

In [1]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import TwoHop
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.pipeline.GraphClassificationPipeline import GraphClassificationPipeline

import torch
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

transform = None
# transform = AddRandomEdges(min_density=0.4)
# transform = ReduceDiameter(max_diam=26)
# transform = TwoHop()

batch = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/mutag', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/collab', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)
print(f"Number of graphs: {len(batch)}")

val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

print("Creating model...")
model = GCN(num_classes=batch.num_classes, 
            num_features=batch.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
print("Model created!")

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = GraphClassificationPipeline(
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.train_validate(
    train_dloader=train_dloader,
    val_dloader=val_dloader,
    epochs=50,
    log=True
)

Number of graphs: 1113
Creating model...
Model created!
Epoch: [0/50]         Training loss: 0.7069         Validation loss: 0.7049         Accuracy: 0.5740
Epoch: [1/50]         Training loss: 0.6760         Validation loss: 0.6850         Accuracy: 0.5695
Epoch: [2/50]         Training loss: 0.6623         Validation loss: 0.6841         Accuracy: 0.5695
Epoch: [3/50]         Training loss: 0.6582         Validation loss: 0.6897         Accuracy: 0.5785
Epoch: [4/50]         Training loss: 0.6698         Validation loss: 0.7252         Accuracy: 0.5695
Epoch: [5/50]         Training loss: 0.6728         Validation loss: 0.6692         Accuracy: 0.5919
Epoch: [6/50]         Training loss: 0.6601         Validation loss: 0.7026         Accuracy: 0.5605
Epoch: [7/50]         Training loss: 0.6670         Validation loss: 0.6630         Accuracy: 0.5785
Epoch: [8/50]         Training loss: 0.6550         Validation loss: 0.6618         Accuracy: 0.5785
Epoch: [9/50]         Training loss

([0.7069067656993866,
  0.676014108317239,
  0.6622718700340816,
  0.6581740230321884,
  0.6697554801191602,
  0.6727905528885978,
  0.6600886540753501,
  0.666987201997212,
  0.6549664310046605,
  0.6479509132249015,
  0.6454114658491952,
  0.6385838857718876,
  0.6342847389834267,
  0.6293637369360242,
  0.6356937118938991,
  0.6230646201542446,
  0.6139719997133527,
  0.6170967561858041,
  0.6155451578753335,
  0.6069285529000419,
  0.5998474189213344,
  0.601343993629728,
  0.5997765575136457,
  0.6029577553272247,
  0.5981541914599282,
  0.6025115890162331,
  0.6044416299888066,
  0.5884752827031272,
  0.5836358283247266,
  0.5837774191583905,
  0.579441059912954,
  0.5754318535327911,
  0.579894802400044,
  0.5729251482657024,
  0.5855588018894196,
  0.5732604478086744,
  0.5719455480575562,
  0.5805698015860149,
  0.5639822781085968,
  0.562117161495345,
  0.5585454191480365,
  0.5618557780981064,
  0.5621865711041859,
  0.5653896714959826,
  0.5568545801298959,
  0.548797036920

In [1]:
import logging
from torch_geometric import datasets
from rewiring.TrainValTestSplitter import TrainValTestSplitter

logging.basicConfig(level=logging.DEBUG)

dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer')
TrainValTestSplitter.single_graph_split(dset=dset, val_size=0.2, test_size=1)

Processing...
Done!
DEBUG:rewiring.TrainValTestSplitter:Proceeding with single graph splitting method
DEBUG:rewiring.TrainValTestSplitter:Train mask: [ 760  396 2798 ... 2072  188  452]
DEBUG:rewiring.TrainValTestSplitter:Val mask: [1551  413  434 1642 3042 2467 1707 2297  274 3238  844 1193  835 2688
 2463 2319 2032  547 1150 1260 3040 1032 1231  796  871 3133  590 2092
 1126 2806  901 1203 2655 2336 1227 1247 1965 1834  611 3314 3252 2555
 3274  415  538 3061 2367 3257 1662 2814 2864 1147 1225  495  169 1762
 2450 2866 2890  578 3273 2982 2193 2604 1048 2098 2431 1559 1749 1818
 1191 1223 2361 2937 2186 2876 1477   82 1796 2014 2337  257 2358 1159
 2415 2488 1194 1789   61  389 2203   31  744 1827 3322 2071 1815 1263
  709 1769  797 2458  629  636 3233 2402 3307 2553 1859 1938 2105  622
 1724 1704 2371   94 3116 1022 1856  226 3179 3087  764 2194 2296 2897
 2663 2953 1109  392 2600  151  174 2759 1110 1019  341 1056 1623 2782
 2401 3304  511 1694 1185 1876  382 1415  366 2214 1297  5

(array([ 760,  396, 2798, ..., 2072,  188,  452]),
 array([1551,  413,  434, 1642, 3042, 2467, 1707, 2297,  274, 3238,  844,
        1193,  835, 2688, 2463, 2319, 2032,  547, 1150, 1260, 3040, 1032,
        1231,  796,  871, 3133,  590, 2092, 1126, 2806,  901, 1203, 2655,
        2336, 1227, 1247, 1965, 1834,  611, 3314, 3252, 2555, 3274,  415,
         538, 3061, 2367, 3257, 1662, 2814, 2864, 1147, 1225,  495,  169,
        1762, 2450, 2866, 2890,  578, 3273, 2982, 2193, 2604, 1048, 2098,
        2431, 1559, 1749, 1818, 1191, 1223, 2361, 2937, 2186, 2876, 1477,
          82, 1796, 2014, 2337,  257, 2358, 1159, 2415, 2488, 1194, 1789,
          61,  389, 2203,   31,  744, 1827, 3322, 2071, 1815, 1263,  709,
        1769,  797, 2458,  629,  636, 3233, 2402, 3307, 2553, 1859, 1938,
        2105,  622, 1724, 1704, 2371,   94, 3116, 1022, 1856,  226, 3179,
        3087,  764, 2194, 2296, 2897, 2663, 2953, 1109,  392, 2600,  151,
         174, 2759, 1110, 1019,  341, 1056, 1623, 2782, 2401,